In [1]:
print("Here begins the Bayesian networks notebook.")

Here begins the Bayesian networks notebook.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/life_satisfaction_canadian_survey_bn_clean.csv")

df_bn.head()

,Gender,Marital_status,Household,Age,Edu_level,Gen_health_state,Life_satisfaction,Mental_health_state,Stress_level,Sense_belonging,Weight_state,BMI_18_above,Sleep_apnea,High_BP,High_cholestrol,Diabetic,Fatigue_syndrome,Mood_disorder,Anxiety_disorder,Respiratory_chronic_con,Musculoskeletal_con,Cardiovascular_con,Health_utility_indx,Pain_status,Cannabies_use,Aboriginal_identity,Birth_country,Immigrant,Food_security,Income_source,Total_income
0,2,1.0,2.0,3,3.0,3.0,9.0,3.0,2.0,2.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,1.0,2.0,0.0,1.0,5.0
1,1,1.0,2.0,5,2.0,3.0,4.0,3.0,3.0,3.0,1.0,2.0,1.0,1.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,NaN,1.0,2.0,2.0,1.0,2.0,0.0,2.0,4.0
2,2,2.0,1.0,5,1.0,2.0,7.0,3.0,3.0,2.0,1.0,2.0,2.0,1.0,2.0,1.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0,2.0,1.0,2.0,NaN,2.0,2.0
3,1,2.0,1.0,5,1.0,3.0,8.0,3.0,3.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,0.0,2.0,3.0
4,1,2.0,1.0,4,3.0,5.0,0.0,5.0,4.0,3.0,3.0,2.0,2.0,1.0,2.0,NaN,2.0,2.0,2.0,NaN,1.0,NaN,1.0,2.0,2.0,2.0,1.0,2.0,0.0,2.0,1.0


In [2]:
# Inspect shape, data types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(108252, 31)
Gender                       int64
Marital_status             float64
Household                  float64
Age                          int64
Edu_level                  float64
Gen_health_state           float64
Life_satisfaction          float64
Mental_health_state        float64
Stress_level               float64
Sense_belonging            float64
Weight_state               float64
BMI_18_above               float64
Sleep_apnea                float64
High_BP                    float64
High_cholestrol            float64
Diabetic                   float64
Fatigue_syndrome           float64
Mood_disorder              float64
Anxiety_disorder           float64
Respiratory_chronic_con    float64
Musculoskeletal_con        float64
Cardiovascular_con         float64
Health_utility_indx        float64
Pain_status                float64
Cannabies_use              float64
Aboriginal_identity        float64
Birth_country              float64
Immigrant                  float64
Food_se

In [3]:
# Convert all columns to string (discrete states)
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Gender                     string
Marital_status             string
Household                  string
Age                        string
Edu_level                  string
Gen_health_state           string
Life_satisfaction          string
Mental_health_state        string
Stress_level               string
Sense_belonging            string
Weight_state               string
BMI_18_above               string
Sleep_apnea                string
High_BP                    string
High_cholestrol            string
Diabetic                   string
Fatigue_syndrome           string
Mood_disorder              string
Anxiety_disorder           string
Respiratory_chronic_con    string
Musculoskeletal_con        string
Cardiovascular_con         string
Health_utility_indx        string
Pain_status                string
Cannabies_use              string
Aboriginal_identity        string
Birth_country              string
Immigrant                  string
Food_security              string
Income_source 

In [4]:
# Check the number of unique states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
6,Life_satisfaction,11
3,Age,5
5,Gen_health_state,5
8,Stress_level,5
7,Mental_health_state,5
30,Total_income,5
9,Sense_belonging,4
28,Food_security,4
4,Edu_level,3
10,Weight_state,3


In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pgmpy.estimators import HillClimbSearch, ExpertKnowledge, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [6]:
# Learn an unconstrained Bayesian Network structure
# - bic-d is for discrete data
# - max_indegree limits complexity
df_bn_model = df_bn.copy()
hc = HillClimbSearch(df_bn_model)

learned_dag = hc.estimate(
    scoring_method="bic-d",
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag.nodes()))
print("Number of edges:", len(learned_dag.edges()))
print("\nLearned edges:")
print(sorted(learned_dag.edges()))

Number of nodes: 31
Number of edges: 85

Learned edges:
[('BMI_18_above', 'Aboriginal_identity'), ('BMI_18_above', 'Birth_country'), ('BMI_18_above', 'Cardiovascular_con'), ('BMI_18_above', 'Diabetic'), ('BMI_18_above', 'Edu_level'), ('BMI_18_above', 'Fatigue_syndrome'), ('BMI_18_above', 'Gen_health_state'), ('BMI_18_above', 'Gender'), ('BMI_18_above', 'High_BP'), ('BMI_18_above', 'High_cholestrol'), ('BMI_18_above', 'Income_source'), ('BMI_18_above', 'Musculoskeletal_con'), ('BMI_18_above', 'Sleep_apnea'), ('BMI_18_above', 'Stress_level'), ('BMI_18_above', 'Total_income'), ('BMI_18_above', 'Weight_state'), ('Birth_country', 'Immigrant'), ('Food_security', 'Aboriginal_identity'), ('Food_security', 'Age'), ('Food_security', 'Anxiety_disorder'), ('Food_security', 'BMI_18_above'), ('Food_security', 'Birth_country'), ('Food_security', 'Cannabies_use'), ('Food_security', 'Cardiovascular_con'), ('Food_security', 'Diabetic'), ('Food_security', 'Edu_level'), ('Food_security', 'Fatigue_syndrome

In [7]:
NON_CAUSED = [
    "Gender", "Age", "Birth_country", "Immigrant", "Aboriginal_identity",
    "Marital_status", "Household", "Edu_level", "Income_source", "Total_income"
]

implausible = [e for e in learned_dag.edges() if e[1] in NON_CAUSED]

print(f"Implausible edges in unconstrained BN: {len(implausible)}")
print("\nEdge | Issue")
print("-" * 70)
for src, tgt in sorted(implausible):
    if tgt in ["Gender", "Age", "Birth_country", "Immigrant", "Aboriginal_identity"]:
        reason = "demographic variable cannot be caused"
    elif tgt in ["Marital_status", "Household"]:
        reason = "household/family structure should not be caused by downstream variables"
    else:
        reason = "socioeconomic background variable should not be caused by health/lifestyle"
    print(f"  {src} → {tgt} | {reason}")

Implausible edges in unconstrained BN: 28

Edge | Issue
----------------------------------------------------------------------
  BMI_18_above → Aboriginal_identity | demographic variable cannot be caused
  BMI_18_above → Birth_country | demographic variable cannot be caused
  BMI_18_above → Edu_level | socioeconomic background variable should not be caused by health/lifestyle
  BMI_18_above → Gender | demographic variable cannot be caused
  BMI_18_above → Income_source | socioeconomic background variable should not be caused by health/lifestyle
  BMI_18_above → Total_income | socioeconomic background variable should not be caused by health/lifestyle
  Birth_country → Immigrant | demographic variable cannot be caused
  Food_security → Aboriginal_identity | demographic variable cannot be caused
  Food_security → Age | demographic variable cannot be caused
  Food_security → Birth_country | demographic variable cannot be caused
  Food_security → Edu_level | socioeconomic background variabl

In [8]:
# Convert the learned graph into a discrete Bayesian Network
bn_unconstrained = DiscreteBayesianNetwork(learned_dag.edges())

# Fit CPDs using Bayesian parameter estimation
# BDeu prior provides smoothing and avoids unstable zero probabilities
bn_unconstrained.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_unconstrained.check_model())

Model check: True


In [9]:
# Show the direct parent variables of Life_satisfaction
if "Life_satisfaction" in bn_unconstrained.nodes():
    print("Parents of Life_satisfaction in unconstrained BN:")
    print(list(bn_unconstrained.predecessors("Life_satisfaction")))

# Inspect CPD metadata for the target variable
if "Life_satisfaction" in bn_unconstrained.nodes():
    cpd_ls = bn_unconstrained.get_cpds("Life_satisfaction")
    print("CPD state names for Life_satisfaction:")
    print(cpd_ls.state_names)

Parents of Life_satisfaction in unconstrained BN:
['Musculoskeletal_con', 'Food_security', 'Income_source']
CPD state names for Life_satisfaction:
{'Life_satisfaction': ['0.0', '1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0'], 'Food_security': ['0.0', '1.0', '2.0', '3.0'], 'Income_source': ['1.0', '2.0'], 'Musculoskeletal_con': ['1.0', '2.0']}


In [10]:
# Create inference engine for the unconstrained BN
infer_unconstrained = VariableElimination(bn_unconstrained)  # type: ignore

In [11]:
# Check valid state values for important variables before running inference
for col in [
    "Life_satisfaction",
    "Mental_health_state",
    "Stress_level",
    "Sense_belonging",
    "Gen_health_state",
    "Food_security",
    "Total_income",
    "Work_hours_binned"
]:
    if col in df_bn_model.columns:
        print(f"\n{col}:")
        print(sorted(df_bn_model[col].dropna().unique()))


Life_satisfaction:
['0.0', '1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0']

Mental_health_state:
['1.0', '2.0', '3.0', '4.0', '5.0']

Stress_level:
['1.0', '2.0', '3.0', '4.0', '5.0']

Sense_belonging:
['1.0', '2.0', '3.0', '4.0']

Gen_health_state:
['1.0', '2.0', '3.0', '4.0', '5.0']

Food_security:
['0.0', '1.0', '2.0', '3.0']

Total_income:
['1.0', '2.0', '3.0', '4.0', '5.0']


In [12]:
# Example query 1:
# Probability distribution of Life_satisfaction under good mental health and low stress
q1 = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "1.0",   # excellent
        "Stress_level": "1.0"           # not at all stressful
    },
    show_progress=False
)

print(q1)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0014 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0011 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.4139 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0010 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0020 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0022 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0168 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.0158 |
+---------

In [13]:
# Example query 2:
# Probability distribution of Life_satisfaction under poor mental health and high stress
q2 = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "5.0",   # poor
        "Stress_level": "5.0"           # extremely stressful
    },
    show_progress=False
)

print(q2)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.1103 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0475 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.0217 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0996 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.1272 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.1415 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.2147 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.1162 |
+---------

In [14]:
# Example query 3:
# Compare Life_satisfaction under strong versus weak sense of belonging
q3_strong = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

q3_weak = infer_unconstrained.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "4.0"        # very weak
    },
    show_progress=False
)

print("Life_satisfaction with very strong sense of belonging:")
print(q3_strong)

print("\nLife_satisfaction with very weak sense of belonging:")
print(q3_weak)

Life_satisfaction with very strong sense of belonging:
+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0044 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0022 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.2266 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0040 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0069 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0117 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0541 |
+-------------------------+--------------------------+
| Life_sat

In [15]:
# Summarize key probability mass for easier interpretation
def summarize_life_satisfaction(query_result):
    probs = dict(zip(
        [str(state) for state in query_result.state_names["Life_satisfaction"]],
        query_result.values
    ))

    low = sum(probs.get(str(x), 0) for x in ["0.0", "1.0", "2.0", "3.0", "4.0"])
    medium = sum(probs.get(str(x), 0) for x in ["5.0", "6.0", "7.0"])
    high = sum(probs.get(str(x), 0) for x in ["8.0", "9.0", "10.0"])

    return pd.Series({
        "Low (0-4)": low,
        "Medium (5-7)": medium,
        "High (8-10)": high
    })

print("Query 1 summary:")
print(summarize_life_satisfaction(q1))

print("\nQuery 2 summary:")
print(summarize_life_satisfaction(q2))

print("\nStrong belonging summary:")
print(summarize_life_satisfaction(q3_strong))

print("\nWeak belonging summary:")
print(summarize_life_satisfaction(q3_weak))

Query 1 summary:
Low (0-4)       0.007765
Medium (5-7)    0.089653
High (8-10)     0.902583
dtype: float64

Query 2 summary:
Low (0-4)       0.526088
Medium (5-7)    0.404751
High (8-10)     0.069162
dtype: float64

Strong belonging summary:
Low (0-4)       0.029307
Medium (5-7)    0.244134
High (8-10)     0.726559
dtype: float64

Weak belonging summary:
Low (0-4)       0.042052
Medium (5-7)    0.274414
High (8-10)     0.683534
dtype: float64


In [16]:
# Build a restricted BN with expert constraints
# The goal is to avoid clearly implausible directions and make Life_satisfaction
# more interpretable as an outcome rather than a cause of demographics, socioeconomic
# factors, or health conditions.

forbidden_edges = [
    # Life satisfaction should not cause demographics
    ("Life_satisfaction", "Gender"),
    ("Life_satisfaction", "Age"),
    ("Life_satisfaction", "Marital_status"),
    ("Life_satisfaction", "Household"),
    ("Life_satisfaction", "Birth_country"),
    ("Life_satisfaction", "Immigrant"),
    ("Life_satisfaction", "Aboriginal_identity"),

    # Life satisfaction should not cause socioeconomic variables
    ("Life_satisfaction", "Edu_level"),
    ("Life_satisfaction", "Income_source"),
    ("Life_satisfaction", "Total_income"),
    ("Life_satisfaction", "Food_security"),

    # Life satisfaction should not cause health / well-being predictors
    ("Life_satisfaction", "Gen_health_state"),
    ("Life_satisfaction", "Mental_health_state"),
    ("Life_satisfaction", "Stress_level"),
    ("Life_satisfaction", "Sense_belonging"),
    ("Life_satisfaction", "BMI_18_above"),
    ("Life_satisfaction", "Weight_state"),
    ("Life_satisfaction", "Sleep_apnea"),
    ("Life_satisfaction", "High_BP"),
    ("Life_satisfaction", "High_cholestrol"),
    ("Life_satisfaction", "Diabetic"),
    ("Life_satisfaction", "Fatigue_syndrome"),
    ("Life_satisfaction", "Mood_disorder"),
    ("Life_satisfaction", "Anxiety_disorder"),
    ("Life_satisfaction", "Respiratory_chronic_con"),
    ("Life_satisfaction", "Musculoskeletal_con"),
    ("Life_satisfaction", "Cardiovascular_con"),
    ("Life_satisfaction", "Health_utility_indx"),
    ("Life_satisfaction", "Pain_status"),
    ("Life_satisfaction", "Cannabies_use"),

    # Health / psychosocial states should not determine basic demographics
    ("Mental_health_state", "Gender"),
    ("Mental_health_state", "Age"),
    ("Stress_level", "Gender"),
    ("Stress_level", "Age"),
    ("Sense_belonging", "Gender"),
    ("Sense_belonging", "Age"),
    ("Gen_health_state", "Gender"),
    ("Gen_health_state", "Age"),

    # Health / psychosocial states should not cause socioeconomic background variables
    ("Mental_health_state", "Income_source"),
    ("Mental_health_state", "Total_income"),
    ("Mental_health_state", "Edu_level"),
    ("Mental_health_state", "Food_security"),

    ("Stress_level", "Income_source"),
    ("Stress_level", "Total_income"),
    ("Stress_level", "Edu_level"),
    ("Stress_level", "Food_security"),

    ("Gen_health_state", "Income_source"),
    ("Gen_health_state", "Total_income"),
    ("Gen_health_state", "Edu_level"),
    ("Gen_health_state", "Food_security"),

    ("Sense_belonging", "Income_source"),
    ("Sense_belonging", "Total_income"),
    ("Sense_belonging", "Edu_level"),
]

expert_knowledge = ExpertKnowledge(
    forbidden_edges=forbidden_edges
)

hc_restricted = HillClimbSearch(df_bn_model)

learned_dag_restricted = hc_restricted.estimate(
    scoring_method="bic-d",
    expert_knowledge=expert_knowledge,
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag_restricted.nodes()))
print("Number of edges:", len(learned_dag_restricted.edges()))
print("\nRestricted BN edges:")
print(sorted(learned_dag_restricted.edges()))

Number of nodes: 31
Number of edges: 85

Restricted BN edges:
[('BMI_18_above', 'Aboriginal_identity'), ('BMI_18_above', 'Birth_country'), ('BMI_18_above', 'Cardiovascular_con'), ('BMI_18_above', 'Diabetic'), ('BMI_18_above', 'Edu_level'), ('BMI_18_above', 'Fatigue_syndrome'), ('BMI_18_above', 'Gen_health_state'), ('BMI_18_above', 'Gender'), ('BMI_18_above', 'High_BP'), ('BMI_18_above', 'High_cholestrol'), ('BMI_18_above', 'Income_source'), ('BMI_18_above', 'Musculoskeletal_con'), ('BMI_18_above', 'Sleep_apnea'), ('BMI_18_above', 'Stress_level'), ('BMI_18_above', 'Total_income'), ('BMI_18_above', 'Weight_state'), ('Birth_country', 'Immigrant'), ('Food_security', 'Aboriginal_identity'), ('Food_security', 'Age'), ('Food_security', 'Anxiety_disorder'), ('Food_security', 'BMI_18_above'), ('Food_security', 'Birth_country'), ('Food_security', 'Cannabies_use'), ('Food_security', 'Cardiovascular_con'), ('Food_security', 'Diabetic'), ('Food_security', 'Edu_level'), ('Food_security', 'Fatigue_sy

In [17]:
# Fit the restricted BN
bn_restricted = DiscreteBayesianNetwork(learned_dag_restricted.edges())

bn_restricted.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Restricted model check:", bn_restricted.check_model())

Restricted model check: True


In [18]:
# Inspect direct parents of Life_satisfaction in the restricted BN
if "Life_satisfaction" in bn_restricted.nodes():
    print("Parents of Life_satisfaction in restricted BN:")
    print(list(bn_restricted.predecessors("Life_satisfaction")))

# Inspect CPD metadata for the target variable in the restricted BN
if "Life_satisfaction" in bn_restricted.nodes():
    cpd_ls_restricted = bn_restricted.get_cpds("Life_satisfaction")
    print("Restricted CPD state names for Life_satisfaction:")
    print(cpd_ls_restricted.state_names)

Parents of Life_satisfaction in restricted BN:
['Musculoskeletal_con', 'Food_security', 'Income_source']
Restricted CPD state names for Life_satisfaction:
{'Life_satisfaction': ['0.0', '1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0'], 'Food_security': ['0.0', '1.0', '2.0', '3.0'], 'Income_source': ['1.0', '2.0'], 'Musculoskeletal_con': ['1.0', '2.0']}


In [19]:
# Create inference engine for the restricted BN
infer_restricted = VariableElimination(bn_restricted)  # type: ignore

In [20]:
# Query 4:
# Probability distribution of Life_satisfaction under favorable psychosocial conditions
q4 = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "1.0",   # excellent
        "Gen_health_state": "1.0",      # excellent
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

print(q4)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0034 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0018 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.2367 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0031 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0057 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0095 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0469 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.0457 |
+---------

In [21]:
# Query 5:
# Probability distribution of Life_satisfaction under adverse psychosocial conditions
q5 = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "5.0",   # poor
        "Gen_health_state": "5.0",      # poor
        "Food_security": "3.0"          # severely food insecure
    },
    show_progress=False
)

print(q5)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0406 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0156 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.0807 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0310 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0483 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0667 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.1998 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.1372 |
+---------

In [22]:
# Query 6:
# Compare life satisfaction under strong versus weak sense of belonging in the restricted BN
q6_strong = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

q6_weak = infer_restricted.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "4.0"        # very weak
    },
    show_progress=False
)

print("Restricted BN - Life_satisfaction with very strong sense of belonging:")
print(q6_strong)

print("\nRestricted BN - Life_satisfaction with very weak sense of belonging:")
print(q6_weak)

Restricted BN - Life_satisfaction with very strong sense of belonging:
+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0044 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0022 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.2266 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0040 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0069 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0117 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0541 |
+-------------------------+----------------------

In [23]:
# Summarize the 0-10 Life_satisfaction distribution into low / medium / high groups
def summarize_life_satisfaction(query_result):
    probs = dict(zip(
        [str(state) for state in query_result.state_names["Life_satisfaction"]],
        query_result.values
    ))

    low = sum(probs.get(x, 0) for x in ["0.0", "1.0", "2.0", "3.0", "4.0"])
    medium = sum(probs.get(x, 0) for x in ["5.0", "6.0", "7.0"])
    high = sum(probs.get(x, 0) for x in ["8.0", "9.0", "10.0"])

    return pd.Series({
        "Low (0-4)": low,
        "Medium (5-7)": medium,
        "High (8-10)": high
    })

print("Unconstrained BN - Query 1 summary:")
print(summarize_life_satisfaction(q1))

print("\nUnconstrained BN - Query 2 summary:")
print(summarize_life_satisfaction(q2))

print("\nUnconstrained BN - Strong belonging summary:")
print(summarize_life_satisfaction(q3_strong))

print("\nUnconstrained BN - Weak belonging summary:")
print(summarize_life_satisfaction(q3_weak))

print("\nRestricted BN - Query 4 summary:")
print(summarize_life_satisfaction(q4))

print("\nRestricted BN - Query 5 summary:")
print(summarize_life_satisfaction(q5))

print("\nRestricted BN - Strong belonging summary:")
print(summarize_life_satisfaction(q6_strong))

print("\nRestricted BN - Weak belonging summary:")
print(summarize_life_satisfaction(q6_weak))

Unconstrained BN - Query 1 summary:
Low (0-4)       0.007765
Medium (5-7)    0.089653
High (8-10)     0.902583
dtype: float64

Unconstrained BN - Query 2 summary:
Low (0-4)       0.526088
Medium (5-7)    0.404751
High (8-10)     0.069162
dtype: float64

Unconstrained BN - Strong belonging summary:
Low (0-4)       0.029307
Medium (5-7)    0.244134
High (8-10)     0.726559
dtype: float64

Unconstrained BN - Weak belonging summary:
Low (0-4)       0.042052
Medium (5-7)    0.274414
High (8-10)     0.683534
dtype: float64

Restricted BN - Query 4 summary:
Low (0-4)       0.023624
Medium (5-7)    0.227119
High (8-10)     0.749256
dtype: float64

Restricted BN - Query 5 summary:
Low (0-4)       0.202205
Medium (5-7)    0.498944
High (8-10)     0.298851
dtype: float64

Restricted BN - Strong belonging summary:
Low (0-4)       0.029307
Medium (5-7)    0.244134
High (8-10)     0.726559
dtype: float64

Restricted BN - Weak belonging summary:
Low (0-4)       0.042052
Medium (5-7)    0.274414
High 

In [24]:
# Build a more strongly constrained BN
# Impose stricter directional assumptions:
# demographics -> socioeconomic context -> health / psychosocial variables -> Life_satisfaction

forbidden_edges_strong = [
    # ---------------------------------------------------------
    # 1. Life satisfaction should not cause other variables
    # ---------------------------------------------------------
    ("Life_satisfaction", "Gender"),
    ("Life_satisfaction", "Age"),
    ("Life_satisfaction", "Marital_status"),
    ("Life_satisfaction", "Household"),
    ("Life_satisfaction", "Birth_country"),
    ("Life_satisfaction", "Immigrant"),
    ("Life_satisfaction", "Aboriginal_identity"),

    ("Life_satisfaction", "Edu_level"),
    ("Life_satisfaction", "Income_source"),
    ("Life_satisfaction", "Total_income"),
    ("Life_satisfaction", "Food_security"),

    ("Life_satisfaction", "Gen_health_state"),
    ("Life_satisfaction", "Mental_health_state"),
    ("Life_satisfaction", "Stress_level"),
    ("Life_satisfaction", "Sense_belonging"),
    ("Life_satisfaction", "BMI_18_above"),
    ("Life_satisfaction", "Weight_state"),
    ("Life_satisfaction", "Sleep_apnea"),
    ("Life_satisfaction", "High_BP"),
    ("Life_satisfaction", "High_cholestrol"),
    ("Life_satisfaction", "Diabetic"),
    ("Life_satisfaction", "Fatigue_syndrome"),
    ("Life_satisfaction", "Mood_disorder"),
    ("Life_satisfaction", "Anxiety_disorder"),
    ("Life_satisfaction", "Respiratory_chronic_con"),
    ("Life_satisfaction", "Musculoskeletal_con"),
    ("Life_satisfaction", "Cardiovascular_con"),
    ("Life_satisfaction", "Health_utility_indx"),
    ("Life_satisfaction", "Pain_status"),
    ("Life_satisfaction", "Cannabies_use"),

    # ---------------------------------------------------------
    # 2. Demographics should be upstream
    # Other variables should not cause demographic/background variables
    # ---------------------------------------------------------
    ("Food_security", "Gender"),
    ("Food_security", "Age"),
    ("Food_security", "Birth_country"),
    ("Food_security", "Immigrant"),
    ("Food_security", "Aboriginal_identity"),

    ("Income_source", "Gender"),
    ("Income_source", "Age"),
    ("Income_source", "Birth_country"),
    ("Income_source", "Immigrant"),
    ("Income_source", "Aboriginal_identity"),

    ("Total_income", "Gender"),
    ("Total_income", "Age"),
    ("Total_income", "Birth_country"),
    ("Total_income", "Immigrant"),
    ("Total_income", "Aboriginal_identity"),

    ("Edu_level", "Gender"),
    ("Edu_level", "Age"),
    ("Edu_level", "Birth_country"),
    ("Edu_level", "Immigrant"),
    ("Edu_level", "Aboriginal_identity"),

    ("Mental_health_state", "Gender"),
    ("Mental_health_state", "Age"),
    ("Mental_health_state", "Birth_country"),
    ("Mental_health_state", "Immigrant"),
    ("Mental_health_state", "Aboriginal_identity"),

    ("Gen_health_state", "Gender"),
    ("Gen_health_state", "Age"),
    ("Gen_health_state", "Birth_country"),
    ("Gen_health_state", "Immigrant"),
    ("Gen_health_state", "Aboriginal_identity"),

    ("Stress_level", "Gender"),
    ("Stress_level", "Age"),
    ("Stress_level", "Birth_country"),
    ("Stress_level", "Immigrant"),
    ("Stress_level", "Aboriginal_identity"),

    ("Sense_belonging", "Gender"),
    ("Sense_belonging", "Age"),
    ("Sense_belonging", "Birth_country"),
    ("Sense_belonging", "Immigrant"),
    ("Sense_belonging", "Aboriginal_identity"),

    ("BMI_18_above", "Gender"),
    ("BMI_18_above", "Age"),
    ("BMI_18_above", "Birth_country"),
    ("BMI_18_above", "Immigrant"),
    ("BMI_18_above", "Aboriginal_identity"),

    # ---------------------------------------------------------
    # 3. Socioeconomic variables should not cause core demographics
    # ---------------------------------------------------------
    ("Income_source", "Marital_status"),
    ("Total_income", "Marital_status"),
    ("Food_security", "Marital_status"),

    # ---------------------------------------------------------
    # 4. Health / psychosocial states should not cause socioeconomic basics
    # ---------------------------------------------------------
    ("Mental_health_state", "Income_source"),
    ("Mental_health_state", "Total_income"),
    ("Mental_health_state", "Edu_level"),
    ("Mental_health_state", "Food_security"),

    ("Gen_health_state", "Income_source"),
    ("Gen_health_state", "Total_income"),
    ("Gen_health_state", "Edu_level"),
    ("Gen_health_state", "Food_security"),

    ("Stress_level", "Income_source"),
    ("Stress_level", "Total_income"),
    ("Stress_level", "Edu_level"),
    ("Stress_level", "Food_security"),

    ("Sense_belonging", "Income_source"),
    ("Sense_belonging", "Total_income"),
    ("Sense_belonging", "Edu_level"),

    # ---------------------------------------------------------
    # 5. Chronic conditions should not cause demographics or socioeconomic basics
    # ---------------------------------------------------------
    ("Mood_disorder", "Gender"),
    ("Mood_disorder", "Age"),
    ("Mood_disorder", "Birth_country"),
    ("Mood_disorder", "Immigrant"),
    ("Mood_disorder", "Aboriginal_identity"),
    ("Mood_disorder", "Edu_level"),
    ("Mood_disorder", "Income_source"),
    ("Mood_disorder", "Total_income"),

    ("Anxiety_disorder", "Gender"),
    ("Anxiety_disorder", "Age"),
    ("Anxiety_disorder", "Birth_country"),
    ("Anxiety_disorder", "Immigrant"),
    ("Anxiety_disorder", "Aboriginal_identity"),
    ("Anxiety_disorder", "Edu_level"),
    ("Anxiety_disorder", "Income_source"),
    ("Anxiety_disorder", "Total_income"),

    ("Diabetic", "Gender"),
    ("Diabetic", "Age"),
    ("Diabetic", "Birth_country"),
    ("Diabetic", "Immigrant"),
    ("Diabetic", "Aboriginal_identity"),
    ("Diabetic", "Income_source"),
    ("Diabetic", "Total_income"),

    ("High_BP", "Gender"),
    ("High_BP", "Age"),
    ("High_BP", "Birth_country"),
    ("High_BP", "Immigrant"),
    ("High_BP", "Aboriginal_identity"),
    ("High_BP", "Income_source"),
    ("High_BP", "Total_income"),

    ("High_cholestrol", "Gender"),
    ("High_cholestrol", "Age"),
    ("High_cholestrol", "Birth_country"),
    ("High_cholestrol", "Immigrant"),
    ("High_cholestrol", "Aboriginal_identity"),
    ("High_cholestrol", "Income_source"),
    ("High_cholestrol", "Total_income"),

    ("Respiratory_chronic_con", "Gender"),
    ("Respiratory_chronic_con", "Age"),
    ("Respiratory_chronic_con", "Birth_country"),
    ("Respiratory_chronic_con", "Immigrant"),
    ("Respiratory_chronic_con", "Aboriginal_identity"),
    ("Respiratory_chronic_con", "Income_source"),
    ("Respiratory_chronic_con", "Total_income"),

    ("Musculoskeletal_con", "Gender"),
    ("Musculoskeletal_con", "Age"),
    ("Musculoskeletal_con", "Birth_country"),
    ("Musculoskeletal_con", "Immigrant"),
    ("Musculoskeletal_con", "Aboriginal_identity"),
    ("Musculoskeletal_con", "Income_source"),
    ("Musculoskeletal_con", "Total_income"),

    ("Cardiovascular_con", "Gender"),
    ("Cardiovascular_con", "Age"),
    ("Cardiovascular_con", "Birth_country"),
    ("Cardiovascular_con", "Immigrant"),
    ("Cardiovascular_con", "Aboriginal_identity"),
    ("Cardiovascular_con", "Income_source"),
    ("Cardiovascular_con", "Total_income"),

    # ---------------------------------------------------------
    # 6. Derived burden variables should not cause more basic states
    # ---------------------------------------------------------
    ("Pain_status", "Gen_health_state"),
    ("Pain_status", "Mental_health_state"),
    ("Pain_status", "Stress_level"),

    ("Health_utility_indx", "Gen_health_state"),
    ("Health_utility_indx", "Mental_health_state"),
    ("Health_utility_indx", "Stress_level"),

    ("Fatigue_syndrome", "Gen_health_state"),
    ("Fatigue_syndrome", "Mental_health_state"),
]

expert_knowledge_strong = ExpertKnowledge(
    forbidden_edges=forbidden_edges_strong
)

hc_restricted_strong = HillClimbSearch(df_bn_model)

learned_dag_restricted_strong = hc_restricted_strong.estimate(
    scoring_method="bic-d",
    expert_knowledge=expert_knowledge_strong,
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag_restricted_strong.nodes()))
print("Number of edges:", len(learned_dag_restricted_strong.edges()))
print("\nStrongly restricted BN edges:")
print(sorted(learned_dag_restricted_strong.edges()))

Number of nodes: 31
Number of edges: 85

Strongly restricted BN edges:
[('Aboriginal_identity', 'Age'), ('Aboriginal_identity', 'Marital_status'), ('Age', 'Income_source'), ('Age', 'Musculoskeletal_con'), ('BMI_18_above', 'Cardiovascular_con'), ('BMI_18_above', 'Diabetic'), ('BMI_18_above', 'Edu_level'), ('BMI_18_above', 'Fatigue_syndrome'), ('BMI_18_above', 'Gen_health_state'), ('BMI_18_above', 'Health_utility_indx'), ('BMI_18_above', 'High_BP'), ('BMI_18_above', 'High_cholestrol'), ('BMI_18_above', 'Respiratory_chronic_con'), ('BMI_18_above', 'Sleep_apnea'), ('BMI_18_above', 'Stress_level'), ('BMI_18_above', 'Total_income'), ('BMI_18_above', 'Weight_state'), ('Birth_country', 'Immigrant'), ('Cannabies_use', 'Aboriginal_identity'), ('Cannabies_use', 'Age'), ('Cannabies_use', 'Birth_country'), ('Cannabies_use', 'Food_security'), ('Cannabies_use', 'Gender'), ('Cannabies_use', 'Income_source'), ('Cannabies_use', 'Marital_status'), ('Cannabies_use', 'Musculoskeletal_con'), ('Food_security

In [25]:
unconstrained_edges     = set(learned_dag.edges())
restricted_edges        = set(learned_dag_restricted.edges())
strong_edges            = set(learned_dag_restricted_strong.edges())

shared_all      = unconstrained_edges & restricted_edges & strong_edges
shared_ur       = unconstrained_edges & restricted_edges - strong_edges
removed_r       = unconstrained_edges - restricted_edges
removed_s       = unconstrained_edges - strong_edges
added_r         = restricted_edges - unconstrained_edges
added_s         = strong_edges - unconstrained_edges

print(f"Unconstrained BN     — total edges : {len(unconstrained_edges)}")
print(f"Restricted BN        — total edges : {len(restricted_edges)}")
print(f"Strongly restricted  — total edges : {len(strong_edges)}")
print(f"Shared across all three BNs        : {len(shared_all)}")
print(f"Removed in restricted              : {len(removed_r)}")
print(f"Removed in strongly restricted     : {len(removed_s)}")
print(f"New edges in restricted            : {len(added_r)}")
print(f"New edges in strongly restricted   : {len(added_s)}")

print("\n--- Edges removed in restricted BN ---")
for src, tgt in sorted(removed_r):
    print(f"  {src} → {tgt}")

print("\n--- New edges in restricted BN ---")
for src, tgt in sorted(added_r):
    print(f"  {src} → {tgt}")

print("\n--- Edges stable across all three BNs ---")
for src, tgt in sorted(shared_all):
    print(f"  {src} → {tgt}")

Unconstrained BN     — total edges : 85
Restricted BN        — total edges : 85
Strongly restricted  — total edges : 85
Shared across all three BNs        : 56
Removed in restricted              : 2
Removed in strongly restricted     : 29
New edges in restricted            : 2
New edges in strongly restricted   : 29

--- Edges removed in restricted BN ---
  Life_satisfaction → Health_utility_indx
  Life_satisfaction → Mental_health_state

--- New edges in restricted BN ---
  Mental_health_state → Health_utility_indx
  Musculoskeletal_con → Mental_health_state

--- Edges stable across all three BNs ---
  BMI_18_above → Cardiovascular_con
  BMI_18_above → Diabetic
  BMI_18_above → Edu_level
  BMI_18_above → Fatigue_syndrome
  BMI_18_above → Gen_health_state
  BMI_18_above → High_BP
  BMI_18_above → High_cholestrol
  BMI_18_above → Sleep_apnea
  BMI_18_above → Stress_level
  BMI_18_above → Total_income
  BMI_18_above → Weight_state
  Birth_country → Immigrant
  Food_security → Anxiety_dis

In [26]:
# Fit the strongly constrained BN
bn_restricted_strong = DiscreteBayesianNetwork(learned_dag_restricted_strong.edges())

bn_restricted_strong.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Strongly restricted model check:", bn_restricted_strong.check_model())

Strongly restricted model check: True


In [27]:
# Inspect direct parents of Life_satisfaction in the strongly constrained BN
if "Life_satisfaction" in bn_restricted_strong.nodes():
    print("Parents of Life_satisfaction in strongly constrained BN:")
    print(list(bn_restricted_strong.predecessors("Life_satisfaction")))

# Inspect CPD metadata for Life_satisfaction
if "Life_satisfaction" in bn_restricted_strong.nodes():
    cpd_ls_strong = bn_restricted_strong.get_cpds("Life_satisfaction")
    print("Strongly constrained CPD state names for Life_satisfaction:")
    print(cpd_ls_strong.state_names)

Parents of Life_satisfaction in strongly constrained BN:
['Musculoskeletal_con', 'Food_security', 'Income_source']
Strongly constrained CPD state names for Life_satisfaction:
{'Life_satisfaction': ['0.0', '1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '8.0', '9.0'], 'Food_security': ['0.0', '1.0', '2.0', '3.0'], 'Income_source': ['1.0', '2.0'], 'Musculoskeletal_con': ['1.0', '2.0']}


In [28]:
# Create inference engine for the strongly constrained BN
infer_restricted_strong = VariableElimination(bn_restricted_strong)  # type: ignore

In [29]:
# Query A:
# Probability distribution of Life_satisfaction under favorable psychosocial conditions
q_strong_1 = infer_restricted_strong.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "1.0",   # excellent
        "Gen_health_state": "1.0",      # excellent
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

print(q_strong_1)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0032 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0017 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.2314 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0030 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0054 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0093 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0444 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.0456 |
+---------

In [30]:
# Query B:
# Probability distribution of Life_satisfaction under adverse psychosocial conditions
q_strong_2 = infer_restricted_strong.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Mental_health_state": "5.0",   # poor
        "Gen_health_state": "5.0",      # poor
        "Food_security": "3.0"          # severely food insecure
    },
    show_progress=False
)

print(q_strong_2)

+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0463 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0163 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.0831 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0311 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0509 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0671 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.2071 |
+-------------------------+--------------------------+
| Life_satisfaction(6.0)  |                   0.1328 |
+---------

In [31]:
# Query C:
# Compare Life_satisfaction under strong versus weak sense of belonging
q_strong_belonging = infer_restricted_strong.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "1.0"        # very strong
    },
    show_progress=False
)

q_weak_belonging = infer_restricted_strong.query(
    variables=["Life_satisfaction"],
    evidence={  # type: ignore
        "Sense_belonging": "4.0"        # very weak
    },
    show_progress=False
)

print("Strongly constrained BN - Life_satisfaction with very strong sense of belonging:")
print(q_strong_belonging)

print("\nStrongly constrained BN - Life_satisfaction with very weak sense of belonging:")
print(q_weak_belonging)

Strongly constrained BN - Life_satisfaction with very strong sense of belonging:
+-------------------------+--------------------------+
| Life_satisfaction       |   phi(Life_satisfaction) |
+=========================+==========================+
| Life_satisfaction(0.0)  |                   0.0043 |
+-------------------------+--------------------------+
| Life_satisfaction(1.0)  |                   0.0021 |
+-------------------------+--------------------------+
| Life_satisfaction(10.0) |                   0.2271 |
+-------------------------+--------------------------+
| Life_satisfaction(2.0)  |                   0.0039 |
+-------------------------+--------------------------+
| Life_satisfaction(3.0)  |                   0.0068 |
+-------------------------+--------------------------+
| Life_satisfaction(4.0)  |                   0.0114 |
+-------------------------+--------------------------+
| Life_satisfaction(5.0)  |                   0.0528 |
+-------------------------+------------

In [32]:
# Summarize the 0-10 Life_satisfaction distribution into low / medium / high groups
def summarize_life_satisfaction(query_result):
    probs = dict(zip(
        [str(state) for state in query_result.state_names["Life_satisfaction"]],
        query_result.values
    ))

    low = sum(probs.get(x, 0) for x in ["0.0", "1.0", "2.0", "3.0", "4.0"])
    medium = sum(probs.get(x, 0) for x in ["5.0", "6.0", "7.0"])
    high = sum(probs.get(x, 0) for x in ["8.0", "9.0", "10.0"])

    return pd.Series({
        "Low (0-4)": low,
        "Medium (5-7)": medium,
        "High (8-10)": high
    })

print("Strongly constrained BN - favorable conditions:")
print(summarize_life_satisfaction(q_strong_1))

print("\nStrongly constrained BN - adverse conditions:")
print(summarize_life_satisfaction(q_strong_2))

print("\nStrongly constrained BN - strong belonging:")
print(summarize_life_satisfaction(q_strong_belonging))

print("\nStrongly constrained BN - weak belonging:")
print(summarize_life_satisfaction(q_weak_belonging))

Strongly constrained BN - favorable conditions:
Low (0-4)       0.022683
Medium (5-7)    0.227868
High (8-10)     0.749449
dtype: float64

Strongly constrained BN - adverse conditions:
Low (0-4)       0.211647
Medium (5-7)    0.491465
High (8-10)     0.296888
dtype: float64

Strongly constrained BN - strong belonging:
Low (0-4)       0.02850
Medium (5-7)    0.24214
High (8-10)     0.72936
dtype: float64

Strongly constrained BN - weak belonging:
Low (0-4)       0.040193
Medium (5-7)    0.270375
High (8-10)     0.689432
dtype: float64


In [33]:
# Compare parents of Life_satisfaction across BN versions
comparison_rows = []

for model_name, model_obj in [
    ("Unconstrained", bn_unconstrained),
    ("Restricted", bn_restricted),
    ("Strongly restricted", bn_restricted_strong)
]:
    if "Life_satisfaction" in model_obj.nodes():
        comparison_rows.append({
            "Model": model_name,
            "Parents of Life_satisfaction": list(model_obj.predecessors("Life_satisfaction")),
            "Children of Life_satisfaction": list(model_obj.successors("Life_satisfaction"))
        })

pd.DataFrame(comparison_rows)

,Model,Parents of Life_satisfaction,Children of Life_satisfaction
0,Unconstrained,"[Musculoskeletal_con, Food_security, Income_so...","[Mental_health_state, Health_utility_indx]"
1,Restricted,"[Musculoskeletal_con, Food_security, Income_so...",[]
2,Strongly restricted,"[Musculoskeletal_con, Food_security, Income_so...",[]


In [35]:
import numpy as np
def bic_local_score(node, parents, data):
    n = len(data.dropna(subset=[node] + parents))
    node_states = data[node].dropna().unique()
    r_i = len(node_states)

    if len(parents) == 0:
        q_i = 1
        counts = data[node].value_counts()
        ll = sum(c * np.log(c / counts.sum()) for c in counts if c > 0)
        k = q_i * (r_i - 1)
    else:
        parent_states = data[parents + [node]].dropna()
        grouped = parent_states.groupby(parents)
        q_i = len(grouped)
        ll = 0
        for _, group in grouped:
            total = len(group)
            if total == 0:
                continue
            counts = group[node].value_counts()
            ll += sum(c * np.log(c / total) for c in counts if c > 0)
        k = q_i * (r_i - 1)

    return ll - (k / 2) * np.log(n)

def compute_total_bic(dag, data):
    return sum(
        bic_local_score(node, list(dag.predecessors(node)), data)
        for node in dag.nodes()
    )

score_u = compute_total_bic(bn_unconstrained,      df_bn_model)
score_r = compute_total_bic(bn_restricted,          df_bn_model)
score_s = compute_total_bic(bn_restricted_strong,   df_bn_model)

print(f"BIC score — Unconstrained BN     : {score_u:.2f}")
print(f"BIC score — Restricted BN        : {score_r:.2f}")
print(f"BIC score — Strongly restricted  : {score_s:.2f}")
print()
print(f"Restricted vs Unconstrained      : {score_r - score_u:.2f} ({abs((score_r-score_u)/score_u)*100:.2f}%)")
print(f"Strongly restricted vs Unconstrained : {score_s - score_u:.2f} ({abs((score_s-score_u)/score_u)*100:.2f}%)")
print()
for label, diff in [("Restricted", score_r - score_u), ("Strongly restricted", score_s - score_u)]:
    if diff < 0:
        print(f"{label} BN has lower (worse) BIC — constraints cost {abs(diff/score_u)*100:.2f}% in fit.")
    else:
        print(f"{label} BN has equal or better BIC — constraints did not reduce fit.")

BIC score — Unconstrained BN     : -1418555.58
BIC score — Restricted BN        : -1417765.72
BIC score — Strongly restricted  : -1481407.55

Restricted vs Unconstrained      : 789.86 (0.06%)
Strongly restricted vs Unconstrained : -62851.97 (4.43%)

Restricted BN has equal or better BIC — constraints did not reduce fit.
Strongly restricted BN has lower (worse) BIC — constraints cost 4.43% in fit.


In [36]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_bn_regression(bn_model, df_data, target="Life_satisfaction"):
    infer = VariableElimination(bn_model)
    evidence_cols = [c for c in df_data.columns if c != target]
    df_pred = df_data.dropna(subset=[target]).copy()

    y_true = []
    y_pred = []

    for _, row in df_pred.iterrows():
        evidence = {
            col: str(row[col])
            for col in evidence_cols
            if pd.notna(row[col]) and str(row[col]) not in ["<NA>", "nan", "None"]
        }
        try:
            result = infer.query(
                variables=[target],
                evidence=evidence,
                show_progress=False
            )
            states = result.state_names[target]
            probs  = result.values

            # Expected value as weighted sum of states
            expected = sum(
                float(s) * p for s, p in zip(states, probs)
            )

            y_true.append(float(row[target]))
            y_pred.append(expected)
        except Exception:
            continue

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return {
        "N": len(y_true),
        "MAE":  round(mean_absolute_error(y_true, y_pred), 3),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 3),
        "R²":   round(r2_score(y_true, y_pred), 3),
    }

print("Evaluating unconstrained BN...")
perf_u = evaluate_bn_regression(bn_unconstrained, df_bn_model)
print(perf_u)

print("\nEvaluating restricted BN...")
perf_r = evaluate_bn_regression(bn_restricted, df_bn_model)
print(perf_r)

print("\nEvaluating strongly restricted BN...")
perf_s = evaluate_bn_regression(bn_restricted_strong, df_bn_model)
print(perf_s)

Evaluating unconstrained BN...
{'N': 104052, 'MAE': 1.052, 'RMSE': np.float64(1.402), 'R²': 0.309}

Evaluating restricted BN...
{'N': 104052, 'MAE': 1.213, 'RMSE': np.float64(1.622), 'R²': 0.076}

Evaluating strongly restricted BN...
{'N': 104052, 'MAE': 1.211, 'RMSE': np.float64(1.62), 'R²': 0.078}


In [37]:
comparison = pd.DataFrame([
    {"Model": "Linear Regression",        "Type": "Baseline ML", "CV R²": 0.406, "CV RMSE": 1.300, "CV MAE": 0.961},
    {"Model": "Decision Tree",            "Type": "Baseline ML", "CV R²": 0.389, "CV RMSE": 1.319, "CV MAE": 0.979},
    {"Model": "Random Forest",            "Type": "Baseline ML", "CV R²": 0.407, "CV RMSE": 1.299, "CV MAE": 0.968},
    {"Model": "XGBoost",                  "Type": "Baseline ML", "CV R²": 0.422, "CV RMSE": 1.282, "CV MAE": 0.952},
    {"Model": "RF Selection → LinReg",    "Type": "Hybrid ML",   "CV R²": 0.404, "CV RMSE": 1.303, "CV MAE": 0.963},
    {"Model": "XGB Selection → LinReg",   "Type": "Hybrid ML",   "CV R²": 0.406, "CV RMSE": 1.301, "CV MAE": 0.961},
    {"Model": "Stacking (LR + RF + XGB)", "Type": "Hybrid ML",   "CV R²": 0.423, "CV RMSE": 1.281, "CV MAE": 0.951},
    {"Model": "BN Unconstrained",         "Type": "BN",          "CV R²": perf_u["R²"], "CV RMSE": perf_u["RMSE"], "CV MAE": perf_u["MAE"]},
    {"Model": "BN Restricted",            "Type": "BN",          "CV R²": perf_r["R²"], "CV RMSE": perf_r["RMSE"], "CV MAE": perf_r["MAE"]},
    {"Model": "BN Strongly Restricted",   "Type": "BN",          "CV R²": perf_s["R²"], "CV RMSE": perf_s["RMSE"], "CV MAE": perf_s["MAE"]},
]).set_index("Model")

print(comparison.to_string())

                                 Type  CV R²  CV RMSE  CV MAE
Model                                                        
Linear Regression         Baseline ML  0.406    1.300   0.961
Decision Tree             Baseline ML  0.389    1.319   0.979
Random Forest             Baseline ML  0.407    1.299   0.968
XGBoost                   Baseline ML  0.422    1.282   0.952
RF Selection → LinReg       Hybrid ML  0.404    1.303   0.963
XGB Selection → LinReg      Hybrid ML  0.406    1.301   0.961
Stacking (LR + RF + XGB)    Hybrid ML  0.423    1.281   0.951
BN Unconstrained                   BN  0.309    1.402   1.052
BN Restricted                      BN  0.076    1.622   1.213
BN Strongly Restricted             BN  0.078    1.620   1.211


In [ ]:
## Bayesian Network — Results Summary (Canadian Survey — Life Satisfaction)

### Structure Learning
# Three BNs were learned using Hill-Climb Search with BIC-d scoring:
# - Unconstrained BN: 85 edges, no domain restrictions
# - Restricted BN: 85 edges, forbidden edges preventing Life_satisfaction
#   from causing other variables and health states from causing
#   socioeconomic/demographic variables
# - Strongly restricted BN: 85 edges, comprehensive causal ordering
#   enforcing demographics → socioeconomic → health/psychosocial →
#   Life_satisfaction

### Implausible Edges (Unconstrained BN)
# 28 of 85 edges (33%) were flagged as causally implausible, including
# Food_security → Gender, Income_source → Age, BMI_18_above → Gender,
# and Musculoskeletal_con → Age. The unconstrained learner treats
# socioeconomic and health variables as causes of fixed demographic
# characteristics — a direct consequence of spurious statistical
# associations in observational data. This is the most complex
# dataset in the study and shows the highest absolute number of
# implausible edges, directly confirming RQ4.

### Structural Stability
# 56 of 85 edges (66%) are stable across all three BNs. The stable
# core is dominated by Food_security and Income_source as major hubs
# pointing to most health and psychosocial outcomes, which is
# sociologically plausible. Life_satisfaction consistently receives
# edges from Musculoskeletal_con, Food_security, and Income_source
# across all three BN versions — a robust finding suggesting
# physical health burden and material security are the primary
# direct predictors of life satisfaction in this dataset.

### BIC Score
# - Restricted BN achieves a marginally better BIC than the
#   unconstrained BN (+789.86, 0.06%) — the forbidden edges
#   redirected two implausible edges at no statistical cost,
#   a strong result.
# - Strongly restricted BN incurs a 4.43% BIC penalty, the
#   largest cost across all datasets. This reflects the
#   comprehensive causal ordering imposed — many statistically
#   convenient but directionally ambiguous edges were blocked,
#   forcing the learner into less optimal but more plausible paths.
#   The 4.43% cost is still modest given the scope of constraints.

### Predictive Performance
# The BN evaluation differs structurally from ML evaluation:
# BNs compute expected life satisfaction as a weighted sum over
# the 11-state probability distribution, while ML models predict
# a direct continuous value.

# - Unconstrained BN: R²=0.309, RMSE=1.402, MAE=1.052
# - Restricted BN: R²=0.076, RMSE=1.622, MAE=1.213
# - Strongly restricted BN: R²=0.078, RMSE=1.620, MAE=1.211
# - Best ML (XGBoost): CV R²=0.422, CV RMSE=1.282, CV MAE=0.952

# The unconstrained BN achieves reasonable R² (0.309) relative to
# ML baselines (~0.40), reflecting that its hub-dominated structure
# captures the dominant statistical associations. The dramatic drop
# in R² for both restricted BNs (0.076–0.078) is notable — it
# indicates that the forbidden edges blocked statistically important
# pathways, particularly the direct Life_satisfaction outgoing edges
# that the unconstrained learner relied on. This is an important
# finding: the unconstrained BN's predictive advantage partly
# derives from causally implausible reverse edges.

# Note on evaluation asymmetry: BN metrics reflect fit to the
# full observed dataset, not held-out generalization. ML models
# use 5-fold CV. BN R² values are therefore not directly comparable
# to ML CV R² values and should be interpreted as fit quality
# indicators rather than generalization estimates.

### Key Findings for RQ1-RQ4
# - RQ1: ML models (CV R²: ~0.40–0.42) substantially outperform
#   restricted BNs (R²: ~0.076–0.078) on regression prediction,
#   even accounting for evaluation asymmetry. For continuous
#   wellbeing outcomes, discriminative ML models have a clear
#   predictive advantage over generative BN models.
# - RQ2: BN inference tables provide explicit conditional
#   probability distributions over life satisfaction — the
#   summarize_life_satisfaction function demonstrates this
#   directly. Good mental health + low stress → 90% probability
#   of high satisfaction; poor conditions → 53% probability of
#   low satisfaction. This probabilistic reasoning under
#   uncertainty is not available from ML models.
# - RQ3: All three BNs agree on the direct parents of
#   Life_satisfaction: Musculoskeletal_con, Food_security,
#   Income_source. This is the most structurally stable finding
#   across all datasets — physical health burden and material
#   security are robust direct predictors of life satisfaction
#   regardless of constraint level.
# - RQ4: 28 implausible edges in the unconstrained BN confirm
#   that data-driven structure learning cannot recover causal
#   direction without domain knowledge. The restricted BN
#   achieves better BIC at no statistical cost, demonstrating
#   that expert constraints can improve both plausibility
#   and statistical fit simultaneously — the strongest such
#   result across all three individual-level datasets.

### Notable Observation — Three BN Strategy
# This is the only dataset where three BN versions were compared.
# The near-identical performance of restricted and strongly
# restricted BNs (R²: 0.076 vs 0.078, RMSE: 1.622 vs 1.620)
# suggests that beyond a basic set of causal ordering constraints,
# additional restrictions do not meaningfully change predictive
# behavior — the core causal skeleton is already captured by
# the restricted BN. The strongly restricted BN adds causal
# plausibility at the cost of 4.43% BIC without improving
# prediction, making the restricted BN the preferred model
# for this dataset.